In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

path = kagglehub.competition_download('spaceship-titanic')

print("Path to competition files:", path)

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv
Path to competition files: /kaggle/input/competitions/spaceship-titanic


In [2]:
# 1. Load Data
train = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/train.csv')
test = pd.read_csv('/kaggle/input/competitions/spaceship-titanic/test.csv')
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
train.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [4]:
empty_cell = train.isnull().sum().sum()
total = np.prod(train.shape)
percentage = (empty_cell/total) * 100
print("Empty cells: ", empty_cell)
print("Total cells: ", total)
print("Null percentage: ", percentage )
train.isnull().sum()

Empty cells:  2324
Total cells:  121702
Null percentage:  1.9095824226389047


PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [6]:
# Change data types (still include null values in the column)
train['CryoSleep'] = train['CryoSleep'].astype("boolean")
train['Age'] = train['Age'].astype("Int64")
train['VIP'] = train['VIP'].astype("boolean")


In [7]:
# Unpack the column - Cabin (deck/num/side)
train[["Deck", "CabinNum", "CabinSide"]] = train['Cabin'].str.split("/", expand=True)
train['Deck'] = train['Deck'].astype("category")
train['CabinNum'] = train['CabinNum'].astype("Int64")
train['CabinSide'] = train['CabinSide'].astype("category")

In [8]:
# Unpack the PassengerId to Group and NumGrpMember(number of group memeber)
train[['Group', 'NumGrpMember']] = train['PassengerId'].str.split("_", expand=True)
train['NumGrpMember'] = train['NumGrpMember'].astype("Int64")


In [9]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   PassengerId   8693 non-null   object  
 1   HomePlanet    8492 non-null   object  
 2   CryoSleep     8476 non-null   boolean 
 3   Cabin         8494 non-null   object  
 4   Destination   8511 non-null   object  
 5   Age           8514 non-null   Int64   
 6   VIP           8490 non-null   boolean 
 7   RoomService   8512 non-null   float64 
 8   FoodCourt     8510 non-null   float64 
 9   ShoppingMall  8485 non-null   float64 
 10  Spa           8510 non-null   float64 
 11  VRDeck        8505 non-null   float64 
 12  Name          8493 non-null   object  
 13  Transported   8693 non-null   bool    
 14  Deck          8494 non-null   category
 15  CabinNum      8494 non-null   Int64   
 16  CabinSide     8494 non-null   category
 17  Group         8693 non-null   object  
 18  NumGrpMe

In [10]:
train['Transported'].value_counts(normalize=True)*100


Transported
True     50.362361
False    49.637639
Name: proportion, dtype: float64

In [11]:
train['Deck'].value_counts(normalize=True)*100

Deck
F    32.893807
G    30.127149
E    10.313162
B     9.171180
C     8.794443
D     5.627502
A     3.013892
T     0.058865
Name: proportion, dtype: float64

In [12]:
train['CabinSide'].value_counts(normalize=True)*100

CabinSide
S    50.482694
P    49.517306
Name: proportion, dtype: float64

In [13]:
train['Destination'].value_counts(normalize=True)*100

Destination
TRAPPIST-1e      69.498296
55 Cancri e      21.149101
PSO J318.5-22     9.352603
Name: proportion, dtype: float64

In [14]:
train['TotalSpend'] = train['RoomService'] + train['FoodCourt'] + train['ShoppingMall'] + train['VRDeck'] + train['Spa']
train['TotalSpend'].isnull().sum()

np.int64(908)

In [15]:
# Make sure my assumption is correct - no spending == sleeping
known = train[train["CryoSleep"].notna()]
known["PredictedCryoSleep"] = known["TotalSpend"] == 0
known = train[train["CryoSleep"].notna()].copy()

known["PredictedCryoSleep"] = known["TotalSpend"].eq(0)

accuracy = (known["CryoSleep"] == known["PredictedCryoSleep"]).mean()*100
print(f"{accuracy:.2f}%")

90.36%


/tmp/ipykernel_59/2805346314.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  known["PredictedCryoSleep"] = known["TotalSpend"] == 0


In [16]:
# Fill the missing value in CryoSleep Column
empty_rows = train['CryoSleep'].isnull()
train.loc[empty_rows, 'CryoSleep'] = train.loc[empty_rows, 'TotalSpend'].map(lambda x: True if x == 0 else False)
train['CryoSleep'].isna().sum()

np.int64(0)

In [17]:
# Fill in the missing value in HomePlanet Column - use Group as reference
group_home_planet = train.dropna(subset=['HomePlanet']).groupby('Group')['HomePlanet'].agg(lambda x: x.mode()[0])

miss_value = train['HomePlanet'].isnull()
train.loc[miss_value, 'HomePlanet'] = train.loc[miss_value, 'Group'].map(group_home_planet)
train['HomePlanet'].isna().sum()

np.int64(111)

In [18]:
train['HomePlanet'].value_counts(normalize=True)

HomePlanet
Earth     0.539967
Europa    0.251806
Mars      0.208227
Name: proportion, dtype: float64

In [19]:
# Fill in the missing value in HomePlanet Column - use proportional
still_missing = train['HomePlanet'].isnull()
still_missing_total = still_missing.sum()

proportion = ['Earth', 'Europa', 'Mars']
weights = [0.53996, 0.25181, 0.20823]

train.loc[still_missing, 'HomePlanet'] = np.random.choice(proportion, size=still_missing_total, p=weights)
train['HomePlanet'].isnull().sum()

np.int64(0)

Deck Proportion:
* F    0.328938
* G    0.301271
* E    0.103132
* B    0.091712
* C    0.087944
* D    0.056275
* A    0.030139
* T    0.000589

In [20]:
# Fill missing value for Deck Column
miss_deck = train['Deck'].isnull()
miss_deck_total = miss_deck.sum()

category = ['F', 'G', 'E', 'B', 'C', 'D', 'A', 'T']
weights = [0.329, 0.301, 0.103, 0.092, 0.088, 0.056, 0.030, 0.001]

train.loc[miss_deck, 'Deck'] = np.random.choice(category, size=miss_deck_total, p=weights)
train['Deck'].isnull().sum()

np.int64(0)

In [21]:
#  Fill the Carbin Number Column with -1
train['CabinNum'] = train['CabinNum'].fillna(-1)
train['CabinNum'].isnull().sum()

np.int64(0)

# Fill in missing Cabin Side Value

In [22]:
# Proof - Same group live on the SAME cabin side

# Only look at groups with more than 1 member, where Cabin info is known
known_cabin = train.dropna(subset=['Deck', 'CabinSide'])
group_sizes = known_cabin.groupby('Group').size()
multi_member_groups = group_sizes[group_sizes > 1].index

multi_df = known_cabin[known_cabin['Group'].isin(multi_member_groups)]

# For each such group, check if everyone shares the same Deck and CabinSide
same_deck = multi_df.groupby('Group')['Deck'].nunique() == 1
same_side = multi_df.groupby('Group')['CabinSide'].nunique() == 1

print("Groups sharing same Deck:", same_deck.mean() * 100, "%")
print("Groups sharing same CabinSide:", same_side.mean() * 100, "%")

Groups sharing same Deck: 69.15750915750915 %
Groups sharing same CabinSide: 100.0 %


In [23]:
miss_side = train['CabinSide'].isnull()
group_to_side = train.dropna(subset='CabinSide').groupby('Group')['CabinSide'].first()

train.loc[miss_side, 'CabinSide'] = train.loc[miss_side, 'Group'].map(group_to_side)
train['CabinSide'].isnull().sum()

np.int64(99)

In [24]:
# Fill in missing Cabin Side Value for the second time
probs = train['CabinSide'].value_counts(normalize=True)

missing_side = train['CabinSide'].isnull()
missing_side_total = missing_side.sum()

category = ['S', 'P']

train.loc[missing_side, 'CabinSide'] = np.random.choice(category, size=missing_side_total, p=probs)
train['CabinSide'].isnull().sum()

np.int64(0)

# Fill in missing value in the Age column 
I plan to use regression model here to predict the value.

In [25]:
known_age = train[train['Age'].notna()]
missing_age = train[train['Age'].isnull()]

print(known_age.shape, missing_age.shape)


(8514, 20) (179, 20)


In [26]:
features = ['HomePlanet', 'VIP', 'CryoSleep', 'TotalSpend', 'NumGrpMember', 'Deck', 'CabinSide', 'Destination']

# Apply one-hot encoding
X_known = pd.get_dummies(known_age[features], drop_first=True)
X_missing = pd.get_dummies(missing_age[features], drop_first=True)

# if X_missing had any extra columns not in X_known, they'd get dropped.
X_missing = X_missing.reindex(columns=X_known.columns, fill_value=0)

y_known = known_age['Age']

In [27]:
# Train model with RandomForestRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

X_train, X_val, y_train, y_val = train_test_split(X_known, y_known, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

val_predict = model.predict(X_val)
print("MAE on validation set: ", mean_absolute_error(y_val, val_predict))

MAE on validation set:  10.546066636977605


In [28]:
# Refit all known data and predict the missing values
final_model = RandomForestRegressor(n_estimators=200, random_state=42)
final_model.fit(X_known, y_known)

predicted_age = final_model.predict(X_missing)

missing_age_rows = train['Age'].isnull()

train.loc[missing_age_rows, 'Age'] = predicted_age.round()
train['Age'].astype('Int64')

train['Age'].isnull().sum()


np.int64(0)

# Fill in Missing Value for the VIP Column
Because the VIP is a True/False value - binary category, not a continous number. I am thinking to apply Randome Forest Classfication here to prefict the missing value. We checked earlier that the proportion of VIP and non-VIP are very imbalanced. Only ~2.3% of passengers are VIP. Therefore, I will try to compare the TotalSpend column with the VIP column, maybe I can find some correlation.

In [29]:
train['VIP'].value_counts(normalize=True)*100

VIP
False    97.656066
True      2.343934
Name: proportion, dtype: Float64

In [30]:
# Get the mean of VIP guests' TotalSpend 
train.groupby('VIP')['TotalSpend'].mean()


VIP
False    1411.643705
True     4599.737430
Name: TotalSpend, dtype: float64

The following data tell us if we using the mean of the TotalSpend will misleading the fact. Because there are 1388 guests are non-VIP and they spend more than the threhold amount.

In [31]:
threshold = 2000
print("Non-VIP passengers over threshold:", ((train['VIP'] == False) & (train['TotalSpend'] > threshold)).sum())
print("VIP passengers over threshold:", ((train['VIP'] == True) & (train['TotalSpend'] > threshold)).sum())
print("VIP passengers under threshold:", ((train['VIP'] == True) & (train['TotalSpend'] <= threshold)).sum())

Non-VIP passengers over threshold: 1388
VIP passengers over threshold: 112
VIP passengers under threshold: 67


To use the simple way, I will just fill the empty cell with False, because there are only 2.3% of VIP guests. and the majority of guests are non-VIP

In [32]:
train['VIP'] = train['VIP'].fillna(False)
train['VIP'].isnull().sum()

np.int64(0)

In [33]:
train['Name'].isnull().sum()

np.int64(200)

# Fill in missing values for Name column
Group can be traveling with family memebers, we can look at it from their surname. I decided to just fill in the empty name as "Unknown Unknown" instead of just drop the entire column. 

In [34]:
train['Name'] = train['Name'].fillna("Unknown Unknown")
train['Name'].isnull().sum()

np.int64(0)

# Fill in missing value for Destination Column
I plan to use the group column, to see where their destination are. Fill up the empty destination cells that belong to the same group. Then use propertional strategy to fill in the rest of the empty destination cell.

In [43]:
known_destination = train.dropna(subset=['Destination'])

# Find the most common/ majority destination per group
group_destination = known_destination.groupby('Group')['Destination'].agg(lambda x: x.mode()[0])

# attach each row's group majority destination back onto the row
known_destination['group_destination'] = known_destination['Group'].map(group_destination)

# Create a new column - record boolean - does the majority destination same as the individual destination
known_destination['matches_destination'] = known_destination['Destination'] == known_destination['group_destination']

known_destination['matches_destination'].value_counts(normalize=True)*100

/tmp/ipykernel_59/3732660083.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  known_destination['group_destination'] = known_destination['Group'].map(group_destination)
/tmp/ipykernel_59/3732660083.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  known_destination['matches_destination'] = known_destination['Destination'] == known_destination['group_destination']


matches_destination
True     89.789684
False    10.210316
Name: proportion, dtype: float64

89.8% of group are traveling to the same destination. 
If a passenger's destination is missing, I'll look at where most people in their group are heading and assign that same destination.

In [47]:
unknown_destination = train['Destination'].isnull()

# Rows that have empty value in destination column
unknown_destination_rows = train[unknown_destination]

# Look up their group's majority destination
filled_values = unknown_destination_rows['Group'].map(group_destination)

# assign those filled values back into train, only at the rows where Destination was missing
train.loc[unknown_destination, 'Destination'] = filled_values

train['Destination'].isnull().sum()

np.int64(103)

Still have 103 empty cell under the Destination column. I am going to check is there a special route between the Home Planet and the Destination. So far we have 3 destinations: 
1. TRAPPIST-1e
2. 55 Cancri e
3. PSO J318.5-22

3 Home Planet: 
1. Earth
2. Europa
3. Mars


In [49]:
train.groupby('HomePlanet')['Destination'].value_counts(normalize=True)*100

HomePlanet  Destination  
Earth       TRAPPIST-1e      68.911917
            PSO J318.5-22    15.824698
            55 Cancri e      15.263385
Europa      TRAPPIST-1e      57.149446
            55 Cancri e      41.974170
            PSO J318.5-22     0.876384
Mars        TRAPPIST-1e      85.921788
            55 Cancri e      11.117318
            PSO J318.5-22     2.960894
Name: proportion, dtype: float64

From the data above, I did not find any "special route" between a destination and home planet. However, the data show us that passengers from the same home planet heavily favor certain destinations.


In [51]:
train['Destination'] = train['Destination'].fillna('TRAPPIST-1e')

train['Destination'].isnull().sum()

np.int64(0)

# Start training the model

In [53]:
# Pick the feature columns
drop_columns = ['PassengerId', 'Name', 'Cabin', 'Group', 'Transported']
features = [c for c in train.columns if c not in drop_columns]

X = pd.get_dummies(train[features], drop_first=True)
y = train['Transported']

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)

val_predicted = model.predict(X_val)
print("Validation accuracy: ", accuracy_score(y_val, val_predicted))

Validation accuracy:  0.8090856814261069
